# 3) M2.3 European market design — Exercises

**Goals:** net positions and congestion income, your own multi-zone coupling LP checked against PyPSA, the value of each border's capacity, and who gains and who loses when markets are coupled.

The setup cell holds the lecture's three synthetic zones (SI, HR, HU), the isolated and base transfer capacities, and a PyPSA helper for checking. Flows are positive from the first zone of a border to the second. Prices in €/MWh, quantities in MW, money in €/h.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import pypsa
from scipy.optimize import linprog

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

ZONES = ["SI", "HR", "HU"]
OFFERS = pd.DataFrame([
    ("SI", "nuclear", 10, 700), ("SI", "hydro", 5, 500), ("SI", "coal", 95, 500), ("SI", "gas", 115, 300),
    ("HR", "hydro", 5, 900), ("HR", "wind", 0, 500), ("HR", "gas", 120, 700), ("HR", "ocgt", 160, 400),
    ("HU", "nuclear", 12, 1900), ("HU", "solar", 0, 800), ("HU", "gas", 100, 1500), ("HU", "ocgt", 150, 500),
], columns=["zone", "order", "price", "qty"])
BIDS = pd.DataFrame([
    ("SI", "load", 3000, 1300), ("HR", "load", 3000, 2400), ("HR", "industry", 130, 150),
    ("HU", "load", 3000, 3500), ("HU", "storage", 60, 300),
], columns=["zone", "order", "price", "qty"])
ISOLATED = {("SI", "HR"): 0, ("HR", "HU"): 0, ("SI", "HU"): 0}
BASE_ATC = {("SI", "HR"): 500, ("HR", "HU"): 400, ("SI", "HU"): 300}


def couple_pypsa(atc):
    """Zonal prices and welfare from PyPSA (zones as buses, borders as bidirectional links), for checking."""
    n = pypsa.Network()
    n.set_snapshots([0])
    n.add("Bus", ZONES)
    for r in OFFERS.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, marginal_cost=r.price)
    for r in BIDS.itertuples():
        n.add("Generator", f"{r.zone}_{r.order}", bus=r.zone, p_nom=r.qty, sign=-1, marginal_cost=-r.price)
    for (z0, z1), cap in atc.items():
        n.add("Link", f"{z0}-{z1}", bus0=z0, bus1=z1, p_nom=max(cap, 1e-6), p_min_pu=-1)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n.buses_t.marginal_price.iloc[0], -n.objective

### Warm-ups

**1. Net positions**

Write `net_positions(flows)` for a dict `{(zone0, zone1): flow}`: each zone's net export (MW, positive = exporting). Net positions of a closed system sum to zero.

In [ ]:
def net_positions(flows: dict) -> pd.Series:
    """Net export (MW) of each zone from border flows."""
    ...  # YOUR CODE HERE

In [ ]:
lecture_flows = {("SI", "HR"): 500.0, ("HR", "HU"): -400.0, ("SI", "HU"): -100.0}
pos = net_positions(lecture_flows)
assert np.isclose(pos.sum(), 0) and np.isclose(pos["HR"], -900) and np.isclose(pos["SI"], 400) and np.isclose(pos["HU"], 500)
pos

**2. Congestion income**

Write `congestion_income(prices, flows)` = Σ flow × (price at the receiving end − price at the sending end). Check it on the lecture's base result (18,000 €/h) and on equal prices (0).

In [ ]:
def congestion_income(prices, flows: dict) -> float:
    """Congestion income (EUR/h) of all borders."""
    ...  # YOUR CODE HERE

In [ ]:
base_prices = pd.Series({"SI": 100.0, "HR": 120.0, "HU": 100.0})
assert np.isclose(congestion_income(base_prices, lecture_flows), 18_000)
assert np.isclose(congestion_income(pd.Series(115.0, index=ZONES), lecture_flows), 0), "no price difference, no income"

### Core

**3. Your own coupling LP**

Write `couple(atc)` returning a dict with zonal `prices` (Series), `flows` (dict), `welfare` (€/h), and accepted quantities `sold` and `bought` (arrays in the order of `OFFERS` and `BIDS`). Check it against PyPSA for the isolated case, the base case and five random sets of capacities.

In [ ]:
def couple(atc: dict) -> dict:
    """Multi-zone welfare maximisation with transfer-capacity limits."""
    ...  # YOUR CODE HERE

In [ ]:
rng = np.random.default_rng(3)
cases = [ISOLATED, BASE_ATC] + [{b: float(rng.integers(0, 1200)) for b in BASE_ATC} for _ in range(5)]
for atc in cases:
    mine = couple(atc)
    p_ref, w_ref = couple_pypsa(atc)
    assert np.allclose(mine["prices"].to_numpy(), p_ref.to_numpy()) and np.isclose(mine["welfare"], w_ref), atc
    assert np.isclose(net_positions(mine["flows"]).sum(), 0)
print(f"own coupling LP = PyPSA in all {len(cases)} cases")

**4. What is each border worth?**

Write `border_values(atc, h=0.5)` returning, for every border, the welfare gained per extra MW of its capacity (central finite difference), and compare with the price difference across that border in the base case.

In [ ]:
def border_values(atc: dict, h: float = 0.5) -> pd.DataFrame:
    """Marginal welfare value (EUR/MWh) of each border's capacity, and the absolute price difference across it."""
    ...  # YOUR CODE HERE

In [ ]:
values = border_values(BASE_ATC)
assert np.allclose(values.value_per_MW, values.price_difference, atol=1e-6), "envelope theorem on every border"
assert np.isclose(values.loc["SI-HR", "value_per_MW"], 20) and np.isclose(values.loc["SI-HU", "value_per_MW"], 0)
values.round(3)

**5. Interpretation**

The Slovenia–Hungary border carries power in the base case, yet one more MW on it is worth nothing. Why? And why does the value of the Croatian borders equal the price difference across them — what would a TSO compare this number with before building a new interconnector?

*Your answer here.*

### Challenge

**6. Who gains and who loses from coupling?**

Write `surplus_by_zone(result)` returning, for each zone, consumer surplus and producer surplus (€/h) at that zone's price. Compare isolation with the base coupling: the change in each zone's consumer and producer surplus, and check that the total change plus congestion income equals the welfare gain.

In [ ]:
def surplus_by_zone(result: dict) -> pd.DataFrame:
    """Consumer and producer surplus (EUR/h) per zone at the zonal prices."""
    ...  # YOUR CODE HERE

In [ ]:
iso, cpl = couple(ISOLATED), couple(BASE_ATC)
change = surplus_by_zone(cpl) - surplus_by_zone(iso)
ci = congestion_income(cpl["prices"], cpl["flows"])
assert np.isclose(change.to_numpy().sum() + ci, cpl["welfare"] - iso["welfare"]), "gains + congestion income = welfare gain"
assert change.loc["HR", "consumer"] > 0, "Croatian consumers gain from cheaper imports"
assert change.loc["SI", "consumer"] < 0, "Slovenian consumers pay more after coupling"
assert change.loc["SI", "producer"] > 0 and change.loc["HR", "producer"] < 0
print(f"welfare gain {cpl['welfare'] - iso['welfare']:,.0f} EUR/h, of which congestion income {ci:,.0f} EUR/h")
change.round(0)

**7. Interpretation**

Coupling raises total welfare, but not everyone gains. Describe who wins and who loses in each zone, explain why the losers' losses are smaller than the winners' gains, and say why this distribution matters for the politics of market integration and new interconnectors.

*Your answer here.*